In [1]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame

In [2]:
# Load the OECD tourism data into a data frame.
# We’re interested in the following columns:
# – LOCATION — A three-letter abbreviation for the country name
# – SUBJECT — Either INT_REC (for tourist funds received) or INT-EXP (for tourist expenses). 
# – TIME — A year (integer)
# – VALUE — A float indicating thousands of dollars
df = pd.read_csv(
    '../../../data/oecd_tourism.csv',
    engine = 'pyarrow',
    dtype_backend = 'pyarrow',
    usecols = ['LOCATION', 'SUBJECT', 'TIME', 'Value']
)

df.columns = df.columns.str.upper()

In [3]:
df

,LOCATION,SUBJECT,TIME,VALUE
0,AUS,INT_REC,2008,31159.8
1,AUS,INT_REC,2009,29980.7
2,AUS,INT_REC,2010,35165.5
3,AUS,INT_REC,2011,38710.1
4,AUS,INT_REC,2012,38003.7
...,...,...,...,...
1229,SRB,INT-EXP,2015,1253.644
1230,SRB,INT-EXP,2016,1351.098
1231,SRB,INT-EXP,2017,1549.183
1232,SRB,INT-EXP,2018,1837.317


In [4]:
df['SUBJECT'] = df['SUBJECT'].str.replace('_', '-')
df

,LOCATION,SUBJECT,TIME,VALUE
0,AUS,INT-REC,2008,31159.8
1,AUS,INT-REC,2009,29980.7
2,AUS,INT-REC,2010,35165.5
3,AUS,INT-REC,2011,38710.1
4,AUS,INT-REC,2012,38003.7
...,...,...,...,...
1229,SRB,INT-EXP,2015,1253.644
1230,SRB,INT-EXP,2016,1351.098
1231,SRB,INT-EXP,2017,1549.183
1232,SRB,INT-EXP,2018,1837.317


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1234 entries, 0 to 1233
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype          
---  ------    --------------  -----          
 0   LOCATION  1234 non-null   string[pyarrow]
 1   SUBJECT   1234 non-null   string[pyarrow]
 2   TIME      1234 non-null   int64[pyarrow] 
 3   VALUE     1234 non-null   double[pyarrow]
dtypes: double[pyarrow](1), int64[pyarrow](1), string[pyarrow](2)
memory usage: 41.3 KB


In [6]:
df['TIME'] = df['TIME'].astype('int16')
# A small amount of memory can be saved by optimizing how year VALUEs are stored.

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1234 entries, 0 to 1233
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype          
---  ------    --------------  -----          
 0   LOCATION  1234 non-null   string[pyarrow]
 1   SUBJECT   1234 non-null   string[pyarrow]
 2   TIME      1234 non-null   int16          
 3   VALUE     1234 non-null   double[pyarrow]
dtypes: double[pyarrow](1), int16(1), string[pyarrow](2)
memory usage: 34.0 KB


In [7]:
# Find the five countries that received the greatest amount of tourist dollars, on
# average, across years in the data set.

df.loc[df['SUBJECT'] == 'INT-REC'].groupby('LOCATION')['VALUE'].mean().sort_values().round(1).iloc[-5:]

LOCATION
GBR     51752.1
DEU     53408.6
FRA     65063.3
ESP     69655.8
USA    201613.5
Name: VALUE, dtype: double[pyarrow]

In [8]:
# Find the five countries whose citizens spent the least amount of tourist dollars,
# on average, across years in the data set.

df.loc[df['SUBJECT'] == 'INT-EXP'].groupby('LOCATION')['VALUE'].mean().sort_values().round(1).iloc[:5]

LOCATION
MLT     387.8
CRI     867.1
LVA     919.5
ISL    1072.8
HRV    1115.6
Name: VALUE, dtype: double[pyarrow]

In [9]:
df_oecd_locations = pd.read_csv(
    '../../../data/oecd_locations.csv',
    engine = 'pyarrow',
    dtype_backend = 'pyarrow',
    names = ['LOCATION', 'LAOCATION_FULL_NAME']
)

In [10]:
df_oecd_locations

,LOCATION,LAOCATION_FULL_NAME
0,AUS,Australia
1,AUT,Austria
2,BEL,Belgium
3,CAN,Canada
4,DNK,Denmark
5,FIN,Finland
6,FRA,France
7,DEU,Germany
8,HUN,Hungary
9,ITA,Italy


In [11]:
# Join these two data frames together into a new one. In the new data frame,
# there is no LOCATION column. Instead, there is a name column with the full
# name of the country.
df = df.set_index('LOCATION')
df_oecd_locations = df_oecd_locations.set_index('LOCATION')


df_joined = df.join(df_oecd_locations)

In [12]:
df_joined

,SUBJECT,TIME,VALUE,LAOCATION_FULL_NAME
LOCATION,,,,
AUS,INT-REC,2008,31159.8,Australia
AUS,INT-REC,2009,29980.7,Australia
AUS,INT-REC,2010,35165.5,Australia
AUS,INT-REC,2011,38710.1,Australia
AUS,INT-REC,2012,38003.7,Australia
...,...,...,...,...
SRB,INT-EXP,2015,1253.644,<NA>
SRB,INT-EXP,2016,1351.098,<NA>
SRB,INT-EXP,2017,1549.183,<NA>


In [13]:
df_joined.loc[df_joined['SUBJECT'] == 'INT-REC'].groupby('LAOCATION_FULL_NAME')['VALUE'].mean().sort_values().round(1).iloc[-5:]

LAOCATION_FULL_NAME
Italy              44930.2
United Kingdom     51752.1
Germany            53408.6
France             65063.3
United States     201613.5
Name: VALUE, dtype: double[pyarrow]

In [14]:
df_joined.loc[df_joined['SUBJECT'] == 'INT-EXP'].groupby('LAOCATION_FULL_NAME')['VALUE'].mean().sort_values().round(1).iloc[:5]

LAOCATION_FULL_NAME
Hungary     2918.4
Finland     5877.1
Israel      6726.5
Denmark    11326.2
Austria    11934.6
Name: VALUE, dtype: double[pyarrow]

In [15]:
# when using the full names for grouping, I did not get the same result, because there
# are abbreviations, e.g. "SRB" that do not have a corresponding entry in the oecd_locations file.

#### Beyond the exercise_1
##### What happens if you perform the join in the other direction? That is, what if you invoke join on tourism_df, passing it an argument of locations_df? Do you get the same result?

In [16]:
df_joined_2 = df_oecd_locations.join(df)

In [17]:
df_joined_2

,LAOCATION_FULL_NAME,SUBJECT,TIME,VALUE
LOCATION,,,,
AUS,Australia,INT-REC,2008,31159.8
AUS,Australia,INT-REC,2009,29980.7
AUS,Australia,INT-REC,2010,35165.5
AUS,Australia,INT-REC,2011,38710.1
AUS,Australia,INT-REC,2012,38003.7
...,...,...,...,...
ISR,Israel,INT-EXP,2015,7507.0
ISR,Israel,INT-EXP,2016,8210.3
ISR,Israel,INT-EXP,2017,8986.0


In [18]:
df_joined_2.loc[df_joined_2['SUBJECT'] == 'INT-REC'].groupby('LAOCATION_FULL_NAME')['VALUE'].mean().sort_values().round(1).iloc[-5:]

LAOCATION_FULL_NAME
Italy              44930.2
United Kingdom     51752.1
Germany            53408.6
France             65063.3
United States     201613.5
Name: VALUE, dtype: double[pyarrow]

In [19]:
df_joined_2.loc[df_joined_2['SUBJECT'] == 'INT-EXP'].groupby('LAOCATION_FULL_NAME')['VALUE'].mean().sort_values().round(1).iloc[:5]

LAOCATION_FULL_NAME
Hungary     2918.4
Finland     5877.1
Israel      6726.5
Denmark    11326.2
Austria    11934.6
Name: VALUE, dtype: double[pyarrow]

In [20]:
# I find no difference between the two types of joins, because we get the full names from
# the oecd_locations dataframe, and in every case there are NaN values in the full name column after the join.


In [21]:
len(df_oecd_locations.index.unique())

16

In [22]:
len(df.index.unique())

54

#### Beyond the exercise_2
##### Get the mean tourism income per year rather than by country. Do you see any evidence of less tourism income during the time of the Great Recession, which started in 2008?

In [23]:
recession = df.groupby('TIME')[['VALUE']].mean()

recession['GROWTH'] = round(recession['VALUE'] - recession.iloc[0]['VALUE'], 1)

recession

# compared to 2008 a significant recession was observed in the first two years

,VALUE,GROWTH
TIME,,
2008,16431.489356,0.0
2009,14839.505557,-1592.0
2010,15783.149852,-648.3
2011,17559.12162,1127.6
2012,18227.551574,1796.1
2013,19337.897611,2906.4
2014,20994.577037,4563.1
2015,19838.899056,3407.4
2016,20120.865389,3689.4


#### Beyond the exercise_3
##### Reset the index on locations_df such that it has a (default) numeric index and two columns (LOCATION and NAME). Now run join on locations_df, specifying that you want to use the LOCATION column on the caller rather than its index. (The data frame passed as an argument to join will always be joined on its index.)

In [25]:
df_oecd_locations = df_oecd_locations.reset_index()
df_oecd_locations

,LOCATION,LAOCATION_FULL_NAME
0,AUS,Australia
1,AUT,Austria
2,BEL,Belgium
3,CAN,Canada
4,DNK,Denmark
5,FIN,Finland
6,FRA,France
7,DEU,Germany
8,HUN,Hungary
9,ITA,Italy


In [26]:
df

,SUBJECT,TIME,VALUE
LOCATION,,,
AUS,INT-REC,2008,31159.8
AUS,INT-REC,2009,29980.7
AUS,INT-REC,2010,35165.5
AUS,INT-REC,2011,38710.1
AUS,INT-REC,2012,38003.7
...,...,...,...
SRB,INT-EXP,2015,1253.644
SRB,INT-EXP,2016,1351.098
SRB,INT-EXP,2017,1549.183


In [27]:
df_oecd_locations.join(df, on = 'LOCATION')

,LOCATION,LAOCATION_FULL_NAME,SUBJECT,TIME,VALUE
0,AUS,Australia,INT-REC,2008,31159.8
0,AUS,Australia,INT-REC,2009,29980.7
0,AUS,Australia,INT-REC,2010,35165.5
0,AUS,Australia,INT-REC,2011,38710.1
0,AUS,Australia,INT-REC,2012,38003.7
...,...,...,...,...,...
15,ISR,Israel,INT-EXP,2015,7507.0
15,ISR,Israel,INT-EXP,2016,8210.3
15,ISR,Israel,INT-EXP,2017,8986.0
15,ISR,Israel,INT-EXP,2018,9974.7
